## Stable-system Experiments

**Isolating noisy-state attenuation.** We compare filtered innovation training (KAFFEE; in the linear-Gaussian case standard KF) to a naive Gaussian predictive MLE that treats noisy observations as latent states.

1. **Linear damped spiral** — learned spectral radius vs. observation noise.
2. **Van der Pol limit cycle** — limit-cycle rollouts and transported reference variance.

Figures written to `figures/nochaos/`: `exp1_main`, `exp2_polynomial_rollout`, and `exp2_covariance_appendix`.


In [ ]:
from __future__ import annotations

import math
import os
import random
import time
from pathlib import Path
from typing import Callable

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

%matplotlib inline


In [ ]:
SEED = 7
DTYPE = torch.float32

OSC_CFG = {
    "n_train": 256,
    "n_val": 64,
    "seq_len": 70,
    "radius": 0.965,
    "angle": 0.24,
    "proc_std": 0.03,
    "obs_std": 0.90,
    "baseline_epochs": 180,
    "kaffee_epochs": 180,
    "batch_size": 64,
    "lr": 3e-2,
}

VDP_CFG = {
    "mu": 1.6,
    "dt": 0.05,
    "n_train": 96,
    "n_val": 32,
    "seq_len": 240,
    "proc_std": 0.035,
    "obs_std": 0.24,
    "epochs": 110,
    "batch_size": 32,
    "lr": 1.5e-2,
    "rollout_len": 700,
    "rollout_init": [0.2, 0.2],
    "variance_ref_scale": 0.08,
}

DEFAULT_SWEEP_NOISE_LEVELS = [0.20, 0.35, 0.50, 0.70, 0.90, 1.10, 1.30]
SWEEP_SEED_COUNT = 10
LINEAR_SWEEP_EPOCHS = 50
LINEAR_SWEEP_N_TRAIN = 96
LINEAR_SWEEP_N_VAL = 32

NAIVE_LABEL = "naive MLE Objective"
KAFFEE_LABEL = "KAFFEE"
TRUE_LABEL = "True"
STOCHASTIC_PANEL_SAMPLES = 6

# Color palette
PAPER_KAFFEE_COLOR = "#009E73"   # bluish green
PAPER_NAIVE_COLOR  = "#D55E00"   # vermillion
PAPER_TRUTH_COLOR  = "#000000" 
PAPER_DATA_COLOR   = "#7f7f7f"   # gray

# Figure sizes
PAPER_FIGSIZE_FULLWIDTH_1x4   = (13.0, 3.91)
# Stroke widths are thicker than the main text because this canvas is 13 in wide.
# Included at full line, they print at the main-text stroke width.
ROLLOUT_LW = 3.50
ENSEMBLE_LW = 1.95
GUIDE_LW = 1.85
# exp1_sweep.pdf is drawn near print size and included at \linewidth.
SWEEP_LW = 1.85
SWEEP_GUIDE_LW = 1.15
PAPER_FIGSIZE_FULLWIDTH_1x3   = (13.0, 3.7)
PAPER_FIGSIZE_FULLWIDTH_1x2   = (10.95, 4.05)
PAPER_FIGSIZE_HALFWIDTH_SQUARE = (5.5, 4.0)
import importlib.util
_here = Path.cwd()
_repo = _here.parent if (_here / "nochaos.ipynb").exists() else _here
_spec = importlib.util.spec_from_file_location("paper_figure_style", _repo / "figures" / "paper_style.py")
_paper_style = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_paper_style)

def _apply_paper_style(width, include_fraction=1.0):
    plt.rcParams.update(_paper_style.rc_params(width, include_fraction=include_fraction))

_apply_paper_style(13.0, 1.0)


In [ ]:
def set_thread_env(n_threads: int) -> None:
    for var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ[var] = str(n_threads)
    torch.set_num_threads(n_threads)


def set_all_seeds(seed: int) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_tensor(array: np.ndarray, device: torch.device) -> torch.Tensor:
    return torch.tensor(array, dtype=DTYPE, device=device)


def softplus_diag(raw: torch.Tensor, floor: float = 1e-4) -> torch.Tensor:
    return torch.diag(torch.nn.functional.softplus(raw) + floor)


def batched_eye(batch: int, dim: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    return torch.eye(dim, device=device, dtype=dtype).unsqueeze(0).repeat(batch, 1, 1)


def batched_gaussian_nll(resid: torch.Tensor, cov: torch.Tensor) -> torch.Tensor:
    dim = resid.shape[-1]
    cov = 0.5 * (cov + cov.transpose(-1, -2))
    solve = torch.linalg.solve(cov, resid.unsqueeze(-1)).squeeze(-1)
    logdet = torch.logdet(cov)
    return 0.5 * (dim * math.log(2.0 * math.pi) + logdet + (resid * solve).sum(dim=-1))


def minibatches(data: torch.Tensor, batch_size: int, shuffle: bool = True):
    n = data.shape[0]
    indices = torch.randperm(n, device=data.device) if shuffle else torch.arange(n, device=data.device)
    for start in range(0, n, batch_size):
        yield data[indices[start : start + batch_size]]


def make_stable_spiral_matrix(radius: float, angle: float) -> np.ndarray:
    c, s = math.cos(angle), math.sin(angle)
    return radius * np.array([[c, -s], [s, c]], dtype=np.float32)


def simulate_linear_oscillator(
    n_seq: int,
    seq_len: int,
    radius: float,
    angle: float,
    proc_std: float,
    obs_std: float,
    seed: int,
) -> dict[str, np.ndarray]:
    rng = np.random.default_rng(seed)
    A = make_stable_spiral_matrix(radius, angle)
    z = rng.normal(scale=1.0, size=(n_seq, 2)).astype(np.float32)
    states = np.zeros((n_seq, seq_len, 2), dtype=np.float32)
    obs = np.zeros_like(states)
    for t in range(seq_len):
        z = z @ A.T + rng.normal(scale=proc_std, size=z.shape).astype(np.float32)
        y = z + rng.normal(scale=obs_std, size=z.shape).astype(np.float32)
        states[:, t] = z
        obs[:, t] = y
    return {"A_true": A, "states": states, "obs": obs}


def vdp_drift(state: np.ndarray, mu: float) -> np.ndarray:
    x = state[..., 0]
    v = state[..., 1]
    dx = v
    dv = mu * (1.0 - x**2) * v - x
    return np.stack([dx, dv], axis=-1)


def rk4_step(state: np.ndarray, dt: float, drift_fn: Callable[[np.ndarray], np.ndarray]) -> np.ndarray:
    k1 = drift_fn(state)
    k2 = drift_fn(state + 0.5 * dt * k1)
    k3 = drift_fn(state + 0.5 * dt * k2)
    k4 = drift_fn(state + dt * k3)
    return state + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


def simulate_vdp_dataset(
    n_seq: int,
    seq_len: int,
    dt: float,
    mu: float,
    proc_std: float,
    obs_std: float,
    seed: int,
) -> dict[str, np.ndarray]:
    rng = np.random.default_rng(seed)
    states = np.zeros((n_seq, seq_len, 2), dtype=np.float32)
    obs = np.zeros_like(states)
    z = rng.normal(
        loc=np.array([1.5, 0.0], dtype=np.float32),
        scale=np.array([1.0, 0.8], dtype=np.float32),
        size=(n_seq, 2),
    ).astype(np.float32)
    drift = lambda s: vdp_drift(s, mu)
    for t in range(seq_len):
        z = rk4_step(z, dt, drift)
        z = z + rng.normal(scale=proc_std, size=z.shape).astype(np.float32)
        y = z + rng.normal(scale=obs_std, size=z.shape).astype(np.float32)
        states[:, t] = z
        obs[:, t] = y
    return {"states": states, "obs": obs}


In [ ]:
class LinearObjectiveComparisonModel(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        self.A = nn.Parameter(
            torch.eye(2, device=device, dtype=DTYPE) + 0.05 * torch.randn(2, 2, device=device, dtype=DTYPE)
        )
        self.log_q = nn.Parameter(torch.full((2,), -2.0, device=device, dtype=DTYPE))
        self.log_r = nn.Parameter(torch.full((2,), -0.1, device=device, dtype=DTYPE))

    def predictive_loss(self, obs: torch.Tensor) -> torch.Tensor:
        current = obs[:, :-1]
        target = obs[:, 1:]
        A = self.A
        Q = softplus_diag(self.log_q)
        R = softplus_diag(self.log_r)
        mean = current @ A.T
        cov = (A @ R @ A.T + Q + R).unsqueeze(0)
        resid = target - mean
        nll = batched_gaussian_nll(resid.reshape(-1, 2), cov.repeat(resid.numel() // 2, 1, 1))
        return nll.mean()

    def innovation_loss(self, obs: torch.Tensor, p0_scale: float = 1.0) -> torch.Tensor:
        batch, seq_len, dim = obs.shape
        A = self.A
        Q = softplus_diag(self.log_q)
        R = softplus_diag(self.log_r)
        eye = batched_eye(batch, dim, obs.device, obs.dtype)
        mean = obs[:, 0]
        cov = p0_scale * eye
        losses = []
        for t in range(1, seq_len):
            mean_pred = mean @ A.T
            cov_pred = A.unsqueeze(0) @ cov @ A.T + Q.unsqueeze(0)
            cov_pred = 0.5 * (cov_pred + cov_pred.transpose(-1, -2))
            S = cov_pred + R.unsqueeze(0)
            resid = obs[:, t] - mean_pred
            losses.append(batched_gaussian_nll(resid, S))
            K = torch.linalg.solve(S, cov_pred.transpose(-1, -2)).transpose(-1, -2)
            mean = mean_pred + (K @ resid.unsqueeze(-1)).squeeze(-1)
            IK = eye - K
            cov = IK @ cov_pred @ IK.transpose(-1, -2) + K @ R.unsqueeze(0) @ K.transpose(-1, -2)
            cov = 0.5 * (cov + cov.transpose(-1, -2))
        return torch.stack(losses, dim=0).mean()


class PolynomialMap(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        self.A = nn.Parameter(torch.eye(2, device=device, dtype=DTYPE) + 0.03 * torch.randn(2, 2, device=device, dtype=DTYPE))
        self.W = nn.Parameter(0.03 * torch.randn(2, 7, device=device, dtype=DTYPE))
        self.h = nn.Parameter(torch.zeros(2, device=device, dtype=DTYPE))

    def features(self, z: torch.Tensor) -> torch.Tensor:
        x = z[..., 0]
        v = z[..., 1]
        return torch.stack([x * x, x * v, v * v, x**3, (x * x) * v, x * (v * v), v**3], dim=-1)

    def feature_jacobian(self, z: torch.Tensor) -> torch.Tensor:
        x = z[..., 0]
        v = z[..., 1]
        rows = [
            torch.stack([2.0 * x, torch.zeros_like(x)], dim=-1),
            torch.stack([v, x], dim=-1),
            torch.stack([torch.zeros_like(x), 2.0 * v], dim=-1),
            torch.stack([3.0 * x * x, torch.zeros_like(x)], dim=-1),
            torch.stack([2.0 * x * v, x * x], dim=-1),
            torch.stack([v * v, 2.0 * x * v], dim=-1),
            torch.stack([torch.zeros_like(x), 3.0 * v * v], dim=-1),
        ]
        return torch.stack(rows, dim=-2)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return z @ self.A.T + self.features(z) @ self.W.T + self.h

    def jacobian(self, z: torch.Tensor) -> torch.Tensor:
        jf = self.feature_jacobian(z)
        return self.A.unsqueeze(0) + torch.einsum("of,bfd->bod", self.W, jf)


class NonlinearKaffeeModel(nn.Module):
    def __init__(self, transition: nn.Module, device: torch.device):
        super().__init__()
        self.transition = transition
        self.log_q = nn.Parameter(torch.full((2,), -2.7, device=device, dtype=DTYPE))
        self.log_r = nn.Parameter(torch.full((2,), -1.8, device=device, dtype=DTYPE))

    def predictive_loss(self, obs: torch.Tensor) -> torch.Tensor:
        current = obs[:, :-1].reshape(-1, 2)
        target = obs[:, 1:].reshape(-1, 2)
        mean = self.transition(current)
        F = self.transition.jacobian(current)
        Q = softplus_diag(self.log_q).unsqueeze(0)
        R = softplus_diag(self.log_r).unsqueeze(0)
        cov = F @ R @ F.transpose(-1, -2) + Q + R
        cov = 0.5 * (cov + cov.transpose(-1, -2))
        resid = target - mean
        return batched_gaussian_nll(resid, cov).mean()

    def innovation_loss(self, obs: torch.Tensor, p0_scale: float = 0.15) -> torch.Tensor:
        batch, seq_len, dim = obs.shape
        Q = softplus_diag(self.log_q)
        R = softplus_diag(self.log_r)
        eye = batched_eye(batch, dim, obs.device, obs.dtype)
        mean = obs[:, 0]
        cov = p0_scale * eye
        losses = []
        for t in range(1, seq_len):
            mean_pred = self.transition(mean)
            F = self.transition.jacobian(mean)
            cov_pred = F @ cov @ F.transpose(-1, -2) + Q.unsqueeze(0)
            cov_pred = 0.5 * (cov_pred + cov_pred.transpose(-1, -2))
            S = cov_pred + R.unsqueeze(0)
            resid = obs[:, t] - mean_pred
            losses.append(batched_gaussian_nll(resid, S))
            K = torch.linalg.solve(S, cov_pred.transpose(-1, -2)).transpose(-1, -2)
            mean = mean_pred + (K @ resid.unsqueeze(-1)).squeeze(-1)
            IK = eye - K
            cov = IK @ cov_pred @ IK.transpose(-1, -2) + K @ R.unsqueeze(0) @ K.transpose(-1, -2)
            cov = 0.5 * (cov + cov.transpose(-1, -2))
        return torch.stack(losses, dim=0).mean()


In [ ]:
def fit_model(
    model: nn.Module,
    train_tensor: torch.Tensor,
    val_tensor: torch.Tensor,
    loss_fn: Callable[[torch.Tensor], torch.Tensor],
    epochs: int,
    batch_size: int,
    lr: float,
    clip: float | None = None,
) -> dict[str, list[float]]:
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {"train": [], "val": []}
    best_state = None
    best_val = float("inf")
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        batch_losses = []
        for batch in minibatches(train_tensor, batch_size=batch_size, shuffle=True):
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(batch)
            loss.backward()
            if clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            batch_losses.append(float(loss.detach().cpu()))
        scheduler.step()
        model.eval()
        with torch.no_grad():
            val_loss = float(loss_fn(val_tensor).detach().cpu())
        train_loss = float(np.mean(batch_losses))
        history["train"].append(train_loss)
        history["val"].append(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if epoch % max(1, epochs // 8) == 0 or epoch == epochs - 1:
            print(f"epoch {epoch:03d} | train={train_loss:.4f} | val={val_loss:.4f}")
    if best_state is not None:
        model.load_state_dict(best_state)
    history["seconds"] = [time.time() - start_time]
    return history


def linear_rollout(A: torch.Tensor, z0: torch.Tensor, steps: int) -> torch.Tensor:
    traj = [z0]
    z = z0
    for _ in range(steps - 1):
        z = A @ z
        traj.append(z)
    return torch.stack(traj, dim=0)


def rollout_transition(transition: nn.Module, z0: torch.Tensor, steps: int) -> torch.Tensor:
    traj = [z0]
    z = z0
    for _ in range(steps - 1):
        z = transition(z.unsqueeze(0)).squeeze(0)
        traj.append(z)
    return torch.stack(traj, dim=0)


def deterministic_vdp_rollout(z0: np.ndarray, steps: int, dt: float, mu: float) -> np.ndarray:
    drift = lambda s: vdp_drift(s, mu)
    z = np.asarray(z0, dtype=np.float32).copy()
    traj = np.zeros((steps, 2), dtype=np.float32)
    for t in range(steps):
        traj[t] = z
        z = rk4_step(z, dt, drift).astype(np.float32)
    return traj


def deterministic_vdp_flow(states: np.ndarray, steps: int, dt: float, mu: float) -> np.ndarray:
    drift = lambda s: vdp_drift(s, mu)
    z = np.asarray(states, dtype=np.float32).copy()
    for _ in range(steps):
        z = rk4_step(z, dt, drift).astype(np.float32)
    return z


def sample_linear_state_rollouts(
    A: np.ndarray,
    q_diag: np.ndarray,
    z0: np.ndarray,
    steps: int,
    n_samples: int,
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    state = np.repeat(np.asarray(z0, dtype=np.float32).reshape(1, 2), n_samples, axis=0)
    traj = np.zeros((n_samples, steps, 2), dtype=np.float32)
    noise_scale = np.sqrt(np.asarray(q_diag, dtype=np.float32)).reshape(1, 2)
    for t in range(steps):
        traj[:, t] = state
        if t + 1 < steps:
            state = state @ np.asarray(A, dtype=np.float32).T
            state = state + rng.normal(scale=noise_scale, size=state.shape).astype(np.float32)
    return traj


def sample_vdp_state_rollouts(
    z0: np.ndarray,
    steps: int,
    dt: float,
    mu: float,
    proc_std: float,
    n_samples: int,
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    drift = lambda s: vdp_drift(s, mu)
    state = np.repeat(np.asarray(z0, dtype=np.float32).reshape(1, 2), n_samples, axis=0)
    traj = np.zeros((n_samples, steps, 2), dtype=np.float32)
    for t in range(steps):
        traj[:, t] = state
        if t + 1 < steps:
            state = rk4_step(state, dt, drift).astype(np.float32)
            state = state + rng.normal(scale=proc_std, size=state.shape).astype(np.float32)
    return traj


def sample_nonlinear_state_rollouts(
    transition: nn.Module,
    q_diag: np.ndarray,
    z0: np.ndarray,
    steps: int,
    n_samples: int,
    seed: int,
    device: torch.device,
) -> np.ndarray:
    set_all_seeds(seed)
    with torch.no_grad():
        state = to_tensor(np.repeat(np.asarray(z0, dtype=np.float32).reshape(1, 2), n_samples, axis=0), device)
        noise_scale = to_tensor(np.sqrt(np.asarray(q_diag, dtype=np.float32)).reshape(1, 2), device)
        traj = np.zeros((n_samples, steps, 2), dtype=np.float32)
        for t in range(steps):
            traj[:, t] = state.detach().cpu().numpy()
            if t + 1 < steps:
                state = transition(state)
                state = state + torch.randn_like(state) * noise_scale
    return traj


def extract_diag_variance(log_diag: torch.Tensor) -> np.ndarray:
    with torch.no_grad():
        return (torch.nn.functional.softplus(log_diag) + 1e-4).detach().cpu().numpy().astype(np.float32)



In [ ]:
def plot_rollout_ensemble_panel(
    ax: plt.Axes,
    true_samples: np.ndarray,
    naive_samples: np.ndarray,
    kaffee_samples: np.ndarray,
    start_point: np.ndarray,
    x_label: str,
    y_label: str,
    *,
    legend: bool = True,
) -> None:
    for samples, color, label in (
        (true_samples, PAPER_TRUTH_COLOR, TRUE_LABEL.lower()),
        (naive_samples, PAPER_NAIVE_COLOR, NAIVE_LABEL),
        (kaffee_samples, PAPER_KAFFEE_COLOR, KAFFEE_LABEL),
    ):
        for sample in samples[1:]:
            ax.plot(sample[:, 0], sample[:, 1], lw=ENSEMBLE_LW, alpha=0.14, color=color)
        ax.plot(samples[0, :, 0], samples[0, :, 1], lw=ROLLOUT_LW, alpha=0.95, color=color, label=label)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_aspect("equal", adjustable="datalim")
    if legend:
        ax.legend(frameon=False, loc="upper right")



def sample_state_points(states: torch.Tensor, max_points: int, seed: int) -> torch.Tensor:
    if states.ndim == 3:
        flat_states = states.reshape(-1, states.shape[-1])
    else:
        flat_states = states

    if flat_states.shape[0] > max_points:
        rng = np.random.default_rng(seed)
        indices = rng.choice(flat_states.shape[0], size=max_points, replace=False)
        flat_states = flat_states[indices]

    return flat_states


def horizon_transport_trace_from_points(
    model: NonlinearKaffeeModel,
    points: torch.Tensor,
    horizon: int,
    p_scale: float,
    quantiles: tuple[float, float] | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    flat_states = points

    xy = flat_states.detach().cpu().numpy().copy()
    model.eval()
    with torch.no_grad():
        z = flat_states
        eye = batched_eye(z.shape[0], 2, z.device, z.dtype)
        P = p_scale * eye
        for _ in range(horizon):
            F = model.transition.jacobian(z)
            P = F @ P @ F.transpose(-1, -2)
            P = 0.5 * (P + P.transpose(-1, -2))
            z = model.transition(z)
        traces = torch.diagonal(P, dim1=-2, dim2=-1).sum(dim=-1).detach().cpu().numpy()

    if quantiles is not None:
        lo, hi = np.quantile(traces, quantiles)
        traces = np.clip(traces, lo, hi)
    return xy, traces


def flow_transport_trace_from_points(
    states: np.ndarray,
    steps: int,
    dt: float,
    mu: float,
    p_scale: float,
    quantiles: tuple[float, float] | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    xy = np.asarray(states, dtype=np.float32).copy()
    jac = finite_difference_flow_jacobian(xy, steps=steps, dt=dt, mu=mu)
    traces = p_scale * np.sum(jac * jac, axis=(1, 2))
    if quantiles is not None:
        lo, hi = np.quantile(traces, quantiles)
        traces = np.clip(traces, lo, hi)
    return xy, traces



def finite_difference_flow_jacobian(states: np.ndarray, steps: int, dt: float, mu: float, eps: float = 1e-3) -> np.ndarray:
    base = np.asarray(states, dtype=np.float32)
    jac = np.zeros((base.shape[0], 2, 2), dtype=np.float32)
    eye = np.eye(2, dtype=np.float32)
    for dim in range(2):
        plus = deterministic_vdp_flow(base + eps * eye[dim], steps, dt, mu)
        minus = deterministic_vdp_flow(base - eps * eye[dim], steps, dt, mu)
        jac[:, :, dim] = (plus - minus) / (2.0 * eps)
    return jac


def flow_eigenvalues_on_points(states: np.ndarray, steps: int, dt: float, mu: float) -> np.ndarray:
    return np.linalg.eigvals(finite_difference_flow_jacobian(states, steps, dt, mu))


def model_eigenvalues_on_points(transition: nn.Module, states: np.ndarray, device: torch.device) -> np.ndarray:
    with torch.no_grad():
        jac = transition.jacobian(to_tensor(np.asarray(states, dtype=np.float32), device)).detach().cpu().numpy()
    return np.linalg.eigvals(jac)


def freeze_state_dict(model: nn.Module) -> dict[str, torch.Tensor]:
    return {key: value.detach().clone() for key, value in model.state_dict().items()}



In [ ]:

def _panel_label(ax, letter: str, *, y: float = 1.02) -> None:
    ax.text(
        0.0,
        y,
        f"({letter})",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=plt.rcParams["axes.titlesize"],
        fontweight="bold",
        clip_on=False,
    )

def _despine_all(fig) -> None:
    for ax in fig.get_axes():
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

def _uniform_panel_boxes(axes) -> None:
    """Give every panel the same axes rectangle and top edge."""
    fig = axes[0].figure
    fig.canvas.draw()
    boxes = [ax.get_position().frozen() for ax in axes]
    height = max(box.height for box in boxes)
    top = max(box.y1 for box in boxes)
    left = min(box.x0 for box in boxes)
    right = max(box.x1 for box in boxes)
    gap = 0.07
    width = (right - left - gap * (len(axes) - 1)) / len(axes)
    for index, ax in enumerate(axes):
        ax.set_position([left + index * (width + gap), top - height, width, height])
        for spine in ax.spines.values():
            spine.set_linewidth(1.50)

def _frame_eigenvalue_panel(ax, eig_groups) -> None:
    """Keep the unit circle and eigenvalue markers inside the panel."""
    theta = np.linspace(0.0, 2.0 * np.pi, 360)
    xs = [np.cos(theta)]
    ys = [np.sin(theta)]
    for eigs in eig_groups:
        values = np.asarray(eigs, dtype=np.complex128).ravel()
        xs.append(np.real(values))
        ys.append(np.imag(values))
    x = np.concatenate(xs)
    y = np.concatenate(ys)
    ax.figure.canvas.draw()
    bbox = ax.get_window_extent()
    box_aspect = float(bbox.height) / max(float(bbox.width), 1.0)
    reach = max(float(np.max(np.abs(x))), float(np.max(np.abs(y))), 1.0)
    # Keep the unit circle below the in-panel legend.
    half_w = reach / 0.52
    half_h = half_w * box_aspect
    if half_h < reach / 0.52:
        half_h = reach / 0.52
        half_w = half_h / box_aspect
    ax.set_aspect("auto")
    ax.set_xlim(-half_w, half_w)
    ax.set_ylim(-half_h, half_h)
    ax.set_aspect("equal", adjustable="datalim")
def _draw_exp1_sweep(ax, summary_df: pd.DataFrame) -> None:
    short_labels = {KAFFEE_LABEL: "KAFFEE", NAIVE_LABEL: "naive MLE"}
    palette = {KAFFEE_LABEL: PAPER_KAFFEE_COLOR, NAIVE_LABEL: PAPER_NAIVE_COLOR}
    for method, group in summary_df.groupby("method"):
        group = group.sort_values("obs_std")
        name = str(method)
        ax.errorbar(
            group["obs_std"],
            group["spectral_radius_mean"],
            yerr=group["spectral_radius_ci95"],
            marker="o",
            ms=4.2,
            lw=2.4,
            elinewidth=1.1,
            capthick=1.0,
            capsize=2.2,
            color=palette[name],
            label=short_labels[name],
        )
    truth = ax.axhline(OSC_CFG["radius"], color=PAPER_TRUTH_COLOR, ls="--", lw=1.6, label="true radius")
    ax.set_xlim(0.12, 1.38)
    ax.set_ylim(0.0, 1.08)
    ax.set_xlabel("observation noise std. dev.")
    ax.set_ylabel("learned spectral radius")
    handles = {artist.get_label(): artist for artist in ax.get_lines() + ax.containers}
    ax.legend(
        [truth, handles["KAFFEE"], handles["naive MLE"]],
        ["true radius", "KAFFEE", "naive MLE"],
        loc="center left",
        bbox_to_anchor=(0.04, 0.58),
        bbox_transform=ax.transAxes,
        frameon=False,
        handlelength=1.6,
        labelspacing=0.22,
        borderaxespad=0.0,
    )


def plot_exp1_main(
    summary_df: pd.DataFrame,
    A_true: np.ndarray,
    A_predictive: np.ndarray,
    A_kaffee: np.ndarray,
    true_traj: np.ndarray,
    predictive_traj: np.ndarray,
    kaffee_traj: np.ndarray,
    true_stochastic: np.ndarray,
    predictive_stochastic: np.ndarray,
    kaffee_stochastic: np.ndarray,
    start_point: np.ndarray,
    output_png: Path,
    output_pdf: Path,
) -> None:
    _apply_paper_style(PAPER_FIGSIZE_FULLWIDTH_1x4[0], 1.0)
    fig, axes = plt.subplots(1, 4, figsize=PAPER_FIGSIZE_FULLWIDTH_1x4)
    _draw_exp1_sweep(axes[0], summary_df)
    _panel_label(axes[0], "a")

    for matrix, label, color in (
        (A_true, TRUE_LABEL.lower(), PAPER_TRUTH_COLOR),
        (A_predictive, "naive MLE", PAPER_NAIVE_COLOR),
        (A_kaffee, KAFFEE_LABEL, PAPER_KAFFEE_COLOR),
    ):
        eigvals = np.linalg.eigvals(matrix)
        axes[1].scatter(np.real(eigvals), np.imag(eigvals), s=85, label=label, color=color)
    unit_circle = np.linspace(0, 2 * np.pi, 400)
    axes[1].plot(np.cos(unit_circle), np.sin(unit_circle), "k--", lw=GUIDE_LW, alpha=0.5)
    axes[1].axhline(0.0, color="0.7", lw=GUIDE_LW)
    axes[1].axvline(0.0, color="0.7", lw=GUIDE_LW)
    axes[1].set_xlabel("real")
    axes[1].set_ylabel("imag")
    eig_handles = {artist.get_label(): artist for artist in axes[1].collections}
    axes[1].legend(
        [eig_handles["true"], eig_handles["KAFFEE"], eig_handles["naive MLE"]],
        ["true", "KAFFEE", "naive MLE"],
        frameon=False,
        loc="upper right",
        borderpad=0.15,
        handlelength=1.4,
        labelspacing=0.2,
    )
    _panel_label(axes[1], "b")

    axes[2].plot(true_traj[:, 0], true_traj[:, 1], lw=ROLLOUT_LW, color=PAPER_TRUTH_COLOR, label="true")
    axes[2].plot(predictive_traj[:, 0], predictive_traj[:, 1], lw=ROLLOUT_LW, color=PAPER_NAIVE_COLOR, label=NAIVE_LABEL)
    axes[2].plot(kaffee_traj[:, 0], kaffee_traj[:, 1], lw=ROLLOUT_LW, color=PAPER_KAFFEE_COLOR, label=KAFFEE_LABEL)
    axes[2].set_xlabel("x")
    axes[2].set_ylabel("y")
    axes[2].set_aspect("equal", adjustable="datalim")
    _panel_label(axes[2], "c")

    plot_rollout_ensemble_panel(
        axes[3],
        true_stochastic,
        predictive_stochastic,
        kaffee_stochastic,
        start_point,
        "x",
        "y",
        legend=False,
    )
    _panel_label(axes[3], "d")
    plt.tight_layout()
    _despine_all(fig)
    _uniform_panel_boxes(axes)
    for ax in axes:
        for spine in ax.spines.values():
            spine.set_linewidth(plt.rcParams["axes.linewidth"])
    for ax in axes[1:]:
        ax.set_aspect("equal", adjustable="datalim")
    _frame_eigenvalue_panel(
        axes[1],
        [np.linalg.eigvals(A_true), np.linalg.eigvals(A_predictive), np.linalg.eigvals(A_kaffee)],
    )
    fig.savefig(output_png, dpi=180, bbox_inches="tight", pad_inches=0.01)
    fig.savefig(output_pdf, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)


In [ ]:
def plot_exp2_overview(
    hist_mle: dict[str, list[float]],
    hist_kaffee: dict[str, list[float]],
    true_eigs: np.ndarray,
    mle_eigs: np.ndarray,
    kaffee_eigs: np.ndarray,
    true_rollout: np.ndarray,
    mle_rollout: np.ndarray,
    kaffee_rollout: np.ndarray,
    true_stochastic: np.ndarray,
    mle_stochastic: np.ndarray,
    kaffee_stochastic: np.ndarray,
    output_png: Path,
    output_pdf: Path,
) -> None:
    _apply_paper_style(12.60, 1.0)
    fig, axes = plt.subplots(1, 4, figsize=PAPER_FIGSIZE_FULLWIDTH_1x4)
    axes[0].plot(hist_mle["train"], label="naive train", lw=ROLLOUT_LW, color=PAPER_NAIVE_COLOR)
    axes[0].plot(hist_mle["val"],   label="naive val",   lw=ROLLOUT_LW, ls="--", color=PAPER_NAIVE_COLOR)
    axes[0].plot(hist_kaffee["train"], label="KAFFEE train", lw=ROLLOUT_LW, color=PAPER_KAFFEE_COLOR)
    axes[0].plot(hist_kaffee["val"],   label="KAFFEE val",   lw=ROLLOUT_LW, ls="--", color=PAPER_KAFFEE_COLOR)
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("objective")
    axes[0].legend(frameon=False, loc="upper right", borderpad=0.2, handlelength=1.6, labelspacing=0.3)
    _panel_label(axes[0], "a")

    for eigvals, label, color, alpha, size in (
        (true_eigs, TRUE_LABEL.lower(), PAPER_TRUTH_COLOR, 0.28, 22),
        (mle_eigs, "naive MLE", PAPER_NAIVE_COLOR, 0.22, 18),
        (kaffee_eigs, KAFFEE_LABEL, PAPER_KAFFEE_COLOR, 0.22, 18),
    ):
        axes[1].scatter(np.real(eigvals).ravel(), np.imag(eigvals).ravel(), s=size, alpha=alpha, color=color, label=label)
    unit_circle = np.linspace(0, 2 * np.pi, 400)
    axes[1].plot(np.cos(unit_circle), np.sin(unit_circle), "k--", lw=GUIDE_LW, alpha=0.5)
    axes[1].axhline(0.0, color="0.7", lw=GUIDE_LW)
    axes[1].axvline(0.0, color="0.7", lw=GUIDE_LW)
    axes[1].set_xlabel("real")
    axes[1].set_ylabel("imag")
    eig_legend = axes[1].legend(
        frameon=False,
        loc="upper right",
        borderpad=0.15,
        handlelength=1.4,
        labelspacing=0.2,
    )
    for handle in getattr(eig_legend, "legend_handles", ()):
        handle.set_alpha(1.0)
    _panel_label(axes[1], "b")

    rollout_window = min(260, true_rollout.shape[0])
    axes[2].plot(true_rollout[:rollout_window, 0], true_rollout[:rollout_window, 1], lw=ROLLOUT_LW, color=PAPER_TRUTH_COLOR, label="true")
    axes[2].plot(mle_rollout[:rollout_window, 0], mle_rollout[:rollout_window, 1], lw=ROLLOUT_LW, color=PAPER_NAIVE_COLOR, label=NAIVE_LABEL)
    axes[2].plot(kaffee_rollout[:rollout_window, 0], kaffee_rollout[:rollout_window, 1], lw=ROLLOUT_LW, color=PAPER_KAFFEE_COLOR, label=KAFFEE_LABEL)
    axes[2].set_xlabel("x")
    axes[2].set_ylabel("v")
    axes[2].set_aspect("equal", adjustable="datalim")
    _panel_label(axes[2], "c")

    plot_rollout_ensemble_panel(
        axes[3],
        true_stochastic,
        mle_stochastic,
        kaffee_stochastic,
        true_rollout[0],
        "x",
        "v",
        legend=False,
    )
    _panel_label(axes[3], "d")

    plt.tight_layout()
    _despine_all(fig)
    _uniform_panel_boxes(axes)
    for ax in axes[1:]:
        ax.set_aspect("equal", adjustable="datalim")
    _frame_eigenvalue_panel(axes[1], [true_eigs, mle_eigs, kaffee_eigs])
    fig.savefig(output_png, dpi=180, bbox_inches="tight", pad_inches=0.01)
    fig.savefig(output_pdf, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)
def plot_exp2_covariance_appendix(
    true_cycle: np.ndarray,
    true_var1_xy: np.ndarray,
    true_var1_trace: np.ndarray,
    mle_var1_xy: np.ndarray,
    mle_var1_trace: np.ndarray,
    kaffee_var1_xy: np.ndarray,
    kaffee_var1_trace: np.ndarray,
    true_var20_xy: np.ndarray,
    true_var20_trace: np.ndarray,
    mle_var20_xy: np.ndarray,
    mle_var20_trace: np.ndarray,
    kaffee_var20_xy: np.ndarray,
    kaffee_var20_trace: np.ndarray,
    output_png: Path,
    output_pdf: Path,
) -> None:
    rows = (
        (
            (true_var1_xy, true_var1_trace),
            (mle_var1_xy, mle_var1_trace),
            (kaffee_var1_xy, kaffee_var1_trace),
            "viridis",
            r"$\mathrm{tr}(\Sigma_1)$",
        ),
        (
            (true_var20_xy, true_var20_trace),
            (mle_var20_xy, mle_var20_trace),
            (kaffee_var20_xy, kaffee_var20_trace),
            "inferno",
            r"$\mathrm{tr}(\Sigma_{20})$",
        ),
    )
    titles = ("True Jacobian", "Naive MLE", KAFFEE_LABEL)
    # Two-row paper box: full line by the Lorenz-96 on-page aspect.
    fig_w, fig_h = 12.2, 5.85
    # Fonts scale with the included PDF width, not the canvas, so printed
    # labels match the other full-line figures.
    _apply_paper_style(12.74, 1.0)
    fig = plt.figure(figsize=(fig_w, fig_h))
    grid = fig.add_gridspec(
        2, 4,
        width_ratios=[1, 1, 1, 0.045],
        left=0.055, right=0.985, bottom=0.09, top=0.90,
        wspace=0.16, hspace=0.28,
    )
    axes = np.empty((2, 3), dtype=object)
    for row in range(2):
        for col in range(3):
            axes[row, col] = fig.add_subplot(grid[row, col])
    span = np.concatenate(
        [true_cycle, true_var1_xy, mle_var1_xy, kaffee_var1_xy, true_var20_xy, mle_var20_xy, kaffee_var20_xy],
        axis=0,
    )
    lo = span.min(axis=0)
    hi = span.max(axis=0)
    pad = 0.06 * np.maximum(hi - lo, 1e-6)
    for row, (panels, cmap, cbar_label) in enumerate(
        (
            (rows[0][:3], rows[0][3], rows[0][4]),
            (rows[1][:3], rows[1][3], rows[1][4]),
        )
    ):
        values = np.concatenate([trace for _xy, trace in panels], axis=0)
        vmin = float(np.quantile(values, 0.02))
        vmax = float(np.quantile(values, 0.98))
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
            vmin = float(np.min(values))
            vmax = float(np.max(values))
        if vmax <= vmin:
            vmax = vmin + 1e-6
        norm = mcolors.PowerNorm(gamma=0.8, vmin=vmin, vmax=vmax)
        sc = None
        for col, (xy, trace) in enumerate(panels):
            ax = axes[row, col]
            sc = ax.scatter(xy[:, 0], xy[:, 1], c=trace, s=18, cmap=cmap, norm=norm, linewidths=0)
            ax.set_xlim(float(lo[0] - pad[0]), float(hi[0] + pad[0]))
            ax.set_ylim(float(lo[1] - pad[1]), float(hi[1] + pad[1]))
            ax.set_xticks([-2.5, 2.5])
            ax.set_yticks([-2.5, 2.5])
            ax.tick_params(labelbottom=(row == 1), labelleft=(col == 0))
            if row == 0:
                ax.set_title(titles[col], pad=4)
        cbar = fig.colorbar(sc, cax=fig.add_subplot(grid[row, 3]))
        cbar.ax.tick_params(labelsize=plt.rcParams["ytick.labelsize"])
        cbar.set_label(cbar_label, fontsize=plt.rcParams["axes.labelsize"])
    for ax in axes.ravel():
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    fig.canvas.draw()
    top_left = axes[0, 0].get_position()
    bottom_left = axes[1, 0].get_position()
    bottom_right = axes[1, 2].get_position()
    fig.text(
        top_left.x0 - 0.062,
        0.5 * (top_left.y1 + bottom_left.y0),
        "v",
        rotation=90,
        fontsize=plt.rcParams["axes.labelsize"],
        ha="center",
        va="center",
    )
    fig.text(
        0.5 * (bottom_left.x0 + bottom_right.x1),
        bottom_left.y0 - 0.08,
        "x",
        ha="center",
        fontsize=plt.rcParams["axes.labelsize"],
        va="top",
    )
    fig.savefig(output_png, dpi=180, bbox_inches="tight", pad_inches=0.01)
    fig.savefig(output_pdf, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)



In [ ]:
def run_experiment1_main(device: torch.device, output_dir: Path) -> dict:
    set_all_seeds(SEED)
    train = simulate_linear_oscillator(
        n_seq=OSC_CFG["n_train"],
        seq_len=OSC_CFG["seq_len"],
        radius=OSC_CFG["radius"],
        angle=OSC_CFG["angle"],
        proc_std=OSC_CFG["proc_std"],
        obs_std=OSC_CFG["obs_std"],
        seed=SEED,
    )
    val = simulate_linear_oscillator(
        n_seq=OSC_CFG["n_val"],
        seq_len=OSC_CFG["seq_len"],
        radius=OSC_CFG["radius"],
        angle=OSC_CFG["angle"],
        proc_std=OSC_CFG["proc_std"],
        obs_std=OSC_CFG["obs_std"],
        seed=SEED + 1,
    )
    train_t = to_tensor(train["obs"], device)
    val_t = to_tensor(val["obs"], device)

    init_model = LinearObjectiveComparisonModel(device)
    init_state = freeze_state_dict(init_model)
    predictive_model = LinearObjectiveComparisonModel(device)
    predictive_model.load_state_dict(init_state)
    kaffee_model = LinearObjectiveComparisonModel(device)
    kaffee_model.load_state_dict(init_state)

    hist_predictive = fit_model(
        predictive_model,
        train_t,
        val_t,
        loss_fn=predictive_model.predictive_loss,
        epochs=OSC_CFG["baseline_epochs"],
        batch_size=OSC_CFG["batch_size"],
        lr=OSC_CFG["lr"],
    )
    hist_kaffee = fit_model(
        kaffee_model,
        train_t,
        val_t,
        loss_fn=kaffee_model.innovation_loss,
        epochs=OSC_CFG["kaffee_epochs"],
        batch_size=OSC_CFG["batch_size"],
        lr=OSC_CFG["lr"],
        clip=1.0,
    )

    A_true = train["A_true"]
    A_predictive = predictive_model.A.detach().cpu().numpy()
    A_kaffee = kaffee_model.A.detach().cpu().numpy()
    z0_np = np.asarray(val["states"][0, 0], dtype=np.float32)
    z0 = to_tensor(z0_np, device)
    true_traj = val["states"][0, :45]
    predictive_traj = linear_rollout(predictive_model.A.detach(), z0, 45).detach().cpu().numpy()
    kaffee_traj = linear_rollout(kaffee_model.A.detach(), z0, 45).detach().cpu().numpy()
    true_stochastic = sample_linear_state_rollouts(
        A_true,
        np.full(2, OSC_CFG["proc_std"] ** 2, dtype=np.float32),
        z0_np,
        45,
        STOCHASTIC_PANEL_SAMPLES,
        SEED + 40,
    )
    predictive_stochastic = sample_linear_state_rollouts(
        A_predictive,
        extract_diag_variance(predictive_model.log_q),
        z0_np,
        45,
        STOCHASTIC_PANEL_SAMPLES,
        SEED + 41,
    )
    kaffee_stochastic = sample_linear_state_rollouts(
        A_kaffee,
        extract_diag_variance(kaffee_model.log_q),
        z0_np,
        45,
        STOCHASTIC_PANEL_SAMPLES,
        SEED + 42,
    )

    return {
        "A_true": A_true,
        "A_predictive": A_predictive,
        "A_kaffee": A_kaffee,
        "true_traj": true_traj,
        "predictive_traj": predictive_traj,
        "kaffee_traj": kaffee_traj,
        "true_stochastic": true_stochastic,
        "predictive_stochastic": predictive_stochastic,
        "kaffee_stochastic": kaffee_stochastic,
        "start_point": z0_np,
    }


In [ ]:
def aggregate_sweep(raw_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (obs_std, method), group in raw_df.groupby(["obs_std", "method"]):
        n = len(group)
        spec_std = float(group["spectral_radius"].std(ddof=1)) if n > 1 else 0.0
        fro_std = float(group["fro_error"].std(ddof=1)) if n > 1 else 0.0
        spec_sem = spec_std / math.sqrt(n) if n > 0 else 0.0
        fro_sem = fro_std / math.sqrt(n) if n > 0 else 0.0
        rows.append(
            {
                "obs_std": float(obs_std),
                "method": method,
                "n_seeds": n,
                "spectral_radius_mean": float(group["spectral_radius"].mean()),
                "spectral_radius_std": spec_std,
                "spectral_radius_sem": spec_sem,
                "spectral_radius_ci95": 1.96 * spec_sem,
                "fro_error_mean": float(group["fro_error"].mean()),
                "fro_error_std": fro_std,
                "fro_error_sem": fro_sem,
                "fro_error_ci95": 1.96 * fro_sem,
            }
        )
    return pd.DataFrame(rows).sort_values(["method", "obs_std"]).reset_index(drop=True)


def run_experiment1_sweep(device: torch.device, output_dir: Path, noise_levels: list[float], n_seeds: int) -> pd.DataFrame:
    rows = []
    for level_idx, obs_std in enumerate(noise_levels):
        for seed_idx in range(n_seeds):
            data_seed = 10_000 + 100 * level_idx + seed_idx
            init_seed = 20_000 + 100 * level_idx + seed_idx
            train = simulate_linear_oscillator(
                n_seq=LINEAR_SWEEP_N_TRAIN,
                seq_len=OSC_CFG["seq_len"],
                radius=OSC_CFG["radius"],
                angle=OSC_CFG["angle"],
                proc_std=OSC_CFG["proc_std"],
                obs_std=obs_std,
                seed=data_seed,
            )
            val = simulate_linear_oscillator(
                n_seq=LINEAR_SWEEP_N_VAL,
                seq_len=OSC_CFG["seq_len"],
                radius=OSC_CFG["radius"],
                angle=OSC_CFG["angle"],
                proc_std=OSC_CFG["proc_std"],
                obs_std=obs_std,
                seed=data_seed + 1,
            )
            train_t = to_tensor(train["obs"], device)
            val_t = to_tensor(val["obs"], device)

            set_all_seeds(init_seed)
            init_model = LinearObjectiveComparisonModel(device)
            init_state = freeze_state_dict(init_model)
            predictive_model = LinearObjectiveComparisonModel(device)
            predictive_model.load_state_dict(init_state)
            kaffee_model = LinearObjectiveComparisonModel(device)
            kaffee_model.load_state_dict(init_state)

            fit_model(
                predictive_model,
                train_t,
                val_t,
                predictive_model.predictive_loss,
                epochs=LINEAR_SWEEP_EPOCHS,
                batch_size=64,
                lr=OSC_CFG["lr"],
            )
            fit_model(
                kaffee_model,
                train_t,
                val_t,
                kaffee_model.innovation_loss,
                epochs=LINEAR_SWEEP_EPOCHS,
                batch_size=64,
                lr=OSC_CFG["lr"],
                clip=1.0,
            )

            for label, matrix in (
                (NAIVE_LABEL, predictive_model.A.detach().cpu().numpy()),
                (KAFFEE_LABEL, kaffee_model.A.detach().cpu().numpy()),
            ):
                rows.append(
                    {
                        "obs_std": float(obs_std),
                        "seed": int(seed_idx),
                        "method": label,
                        "spectral_radius": float(np.max(np.abs(np.linalg.eigvals(matrix)))),
                        "fro_error": float(np.linalg.norm(matrix - train["A_true"])),
                    }
                )

    raw_df = pd.DataFrame(rows).sort_values(["obs_std", "seed", "method"]).reset_index(drop=True)
    return aggregate_sweep(raw_df)


In [ ]:
def run_experiment2_polynomial(device: torch.device, output_dir: Path) -> None:
    set_all_seeds(SEED)
    train = simulate_vdp_dataset(
        n_seq=VDP_CFG["n_train"],
        seq_len=VDP_CFG["seq_len"],
        dt=VDP_CFG["dt"],
        mu=VDP_CFG["mu"],
        proc_std=VDP_CFG["proc_std"],
        obs_std=VDP_CFG["obs_std"],
        seed=SEED,
    )
    val = simulate_vdp_dataset(
        n_seq=VDP_CFG["n_val"],
        seq_len=VDP_CFG["seq_len"],
        dt=VDP_CFG["dt"],
        mu=VDP_CFG["mu"],
        proc_std=VDP_CFG["proc_std"],
        obs_std=VDP_CFG["obs_std"],
        seed=SEED + 1,
    )
    train_t = to_tensor(train["obs"], device)
    val_t = to_tensor(val["obs"], device)
    val_states_t = to_tensor(val["states"], device)

    init_model = NonlinearKaffeeModel(PolynomialMap(device), device).to(device)
    init_state = freeze_state_dict(init_model)

    mle_model = NonlinearKaffeeModel(PolynomialMap(device), device).to(device)
    mle_model.load_state_dict(init_state)
    kaffee_model = NonlinearKaffeeModel(PolynomialMap(device), device).to(device)
    kaffee_model.load_state_dict(init_state)

    hist_mle = fit_model(
        mle_model,
        train_t,
        val_t,
        loss_fn=mle_model.predictive_loss,
        epochs=VDP_CFG["epochs"],
        batch_size=VDP_CFG["batch_size"],
        lr=VDP_CFG["lr"],
        clip=1.0,
    )
    hist_kaffee = fit_model(
        kaffee_model,
        train_t,
        val_t,
        loss_fn=kaffee_model.innovation_loss,
        epochs=VDP_CFG["epochs"],
        batch_size=VDP_CFG["batch_size"],
        lr=VDP_CFG["lr"],
        clip=1.0,
    )

    rollout_init = np.asarray(VDP_CFG["rollout_init"], dtype=np.float32)
    vdp_init = to_tensor(rollout_init, device)
    true_rollout = deterministic_vdp_rollout(rollout_init, VDP_CFG["rollout_len"], VDP_CFG["dt"], VDP_CFG["mu"])
    mle_rollout = rollout_transition(mle_model.transition, vdp_init, VDP_CFG["rollout_len"]).detach().cpu().numpy()
    kaffee_rollout = rollout_transition(kaffee_model.transition, vdp_init, VDP_CFG["rollout_len"]).detach().cpu().numpy()
    true_stochastic = sample_vdp_state_rollouts(
        rollout_init,
        260,
        VDP_CFG["dt"],
        VDP_CFG["mu"],
        VDP_CFG["proc_std"],
        STOCHASTIC_PANEL_SAMPLES,
        SEED + 60,
    )
    mle_stochastic = sample_nonlinear_state_rollouts(
        mle_model.transition,
        extract_diag_variance(mle_model.log_q),
        rollout_init,
        260,
        STOCHASTIC_PANEL_SAMPLES,
        SEED + 61,
        device,
    )
    kaffee_stochastic = sample_nonlinear_state_rollouts(
        kaffee_model.transition,
        extract_diag_variance(kaffee_model.log_q),
        rollout_init,
        260,
        STOCHASTIC_PANEL_SAMPLES,
        SEED + 62,
        device,
    )
    true_cycle = val["states"].reshape(-1, 2)

    var1_points = sample_state_points(val_states_t, max_points=2200, seed=SEED)
    var20_points = sample_state_points(val_states_t, max_points=2200, seed=SEED + 1)
    true_var1_xy, true_var1_trace = flow_transport_trace_from_points(
        var1_points.detach().cpu().numpy(),
        steps=1,
        dt=VDP_CFG["dt"],
        mu=VDP_CFG["mu"],
        p_scale=VDP_CFG["variance_ref_scale"],
        quantiles=(0.02, 0.98),
    )
    true_var20_xy, true_var20_trace = flow_transport_trace_from_points(
        var20_points.detach().cpu().numpy(),
        steps=20,
        dt=VDP_CFG["dt"],
        mu=VDP_CFG["mu"],
        p_scale=VDP_CFG["variance_ref_scale"],
        quantiles=(0.02, 0.98),
    )
    mle_var1_xy, mle_var1_trace = horizon_transport_trace_from_points(
        mle_model,
        var1_points,
        horizon=1,
        p_scale=VDP_CFG["variance_ref_scale"],
        quantiles=(0.02, 0.98),
    )
    kaffee_var1_xy, kaffee_var1_trace = horizon_transport_trace_from_points(
        kaffee_model,
        var1_points,
        horizon=1,
        p_scale=VDP_CFG["variance_ref_scale"],
        quantiles=(0.02, 0.98),
    )
    mle_var20_xy, mle_var20_trace = horizon_transport_trace_from_points(
        mle_model,
        var20_points,
        horizon=20,
        p_scale=VDP_CFG["variance_ref_scale"],
        quantiles=(0.02, 0.98),
    )
    kaffee_var20_xy, kaffee_var20_trace = horizon_transport_trace_from_points(
        kaffee_model,
        var20_points,
        horizon=20,
        p_scale=VDP_CFG["variance_ref_scale"],
        quantiles=(0.02, 0.98),
    )

    eig_points = true_rollout[150:650:4]
    true_eigs = flow_eigenvalues_on_points(eig_points, steps=1, dt=VDP_CFG["dt"], mu=VDP_CFG["mu"])
    mle_eigs = model_eigenvalues_on_points(mle_model.transition, eig_points, device)
    kaffee_eigs = model_eigenvalues_on_points(kaffee_model.transition, eig_points, device)

    rollout_png = output_dir / "exp2_polynomial_rollout.png"
    rollout_pdf = output_dir / "exp2_polynomial_rollout.pdf"
    plot_exp2_overview(
        hist_mle,
        hist_kaffee,
        true_eigs,
        mle_eigs,
        kaffee_eigs,
        true_rollout,
        mle_rollout,
        kaffee_rollout,
        true_stochastic,
        mle_stochastic,
        kaffee_stochastic,
        rollout_png,
        rollout_pdf,
    )
    cov_png = output_dir / "exp2_covariance_appendix.png"
    cov_pdf = output_dir / "exp2_covariance_appendix.pdf"
    plot_exp2_covariance_appendix(
        true_cycle,
        true_var1_xy,
        true_var1_trace,
        mle_var1_xy,
        mle_var1_trace,
        kaffee_var1_xy,
        kaffee_var1_trace,
        true_var20_xy,
        true_var20_trace,
        mle_var20_xy,
        mle_var20_trace,
        kaffee_var20_xy,
        kaffee_var20_trace,
        cov_png,
        cov_pdf,
    )

    for stale_path in (output_dir / "exp2_polynomial_losses.png", output_dir / "exp2_polynomial_losses.pdf"):
        stale_path.unlink(missing_ok=True)


### Run experiments


In [ ]:
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if (_cwd / "nochaos.ipynb").exists() else _cwd
OUTPUT_DIR = REPO_ROOT / "figures" / "nochaos"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
set_thread_env(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}  output={OUTPUT_DIR.relative_to(REPO_ROOT)}")


In [ ]:
print("=== Experiment 1: linear spiral (main) ===")
_t0 = time.perf_counter()
exp1_payload = run_experiment1_main(device, OUTPUT_DIR)
print(f"done in {time.perf_counter() - _t0:.1f}s")


In [ ]:
print("=== Experiment 1: observation-noise sweep (7 levels x 10 seeds) ===")
_t0 = time.perf_counter()
exp1_summary = run_experiment1_sweep(device, OUTPUT_DIR, DEFAULT_SWEEP_NOISE_LEVELS, SWEEP_SEED_COUNT)
plot_exp1_main(
    exp1_summary,
    output_png=OUTPUT_DIR / "exp1_main.png",
    output_pdf=OUTPUT_DIR / "exp1_main.pdf",
    **exp1_payload,
)
print(f"done in {time.perf_counter() - _t0:.1f}s")


In [ ]:
print("=== Experiment 2: Van der Pol (polynomial map) ===")
_t0 = time.perf_counter()
run_experiment2_polynomial(device, OUTPUT_DIR)
print(f"done in {time.perf_counter() - _t0:.1f}s")


### Figures


In [ ]:
from IPython.display import Image, display

paper_figures = (
    "exp1_main.png",
    "exp2_polynomial_rollout.png",
    "exp2_covariance_appendix.png",
)
for name in paper_figures:
    path = OUTPUT_DIR / name
    print(path.name)
    display(Image(filename=str(path)))
